# Data Preprocessing & Feature Engineering

Load, clean, engineer features for subway ridership.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('data/raw/subway_ridership.csv')
df['date'] = pd.to_datetime(df['date'])
print(df.head())
print(df.info())

In [ ]:
# Feature Engineering
df['year'] = df['date'].dt.year
df['quarter'] = df['date'].dt.quarter
df['is_holiday'] = 0  # Simplified
df['lag_7'] = df['ridership'].shift(7)
df['rolling_mean_7'] = df['ridership'].rolling(window=7).mean()

# Encode categoricals
le_day = LabelEncoder()
df['dayofweek_encoded'] = le_day.fit_transform(df['dayofweek'])

# Features and target
features = ['dayofweek_encoded', 'month', 'is_weekend', 'is_peak_season', 'lag_7', 'rolling_mean_7']
X = df[features].dropna()
y = df.loc[X.index, 'ridership']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

os.makedirs('data/processed', exist_ok=True)
X_train.to_csv('data/processed/X_train.csv', index=False)
X_test.to_csv('data/processed/X_test.csv', index=False)
np.save('data/processed/y_train.npy', y_train.values)
np.save('data/processed/y_test.npy', y_test.values)
joblib.dump(le_day, 'models/label_encoder.pkl')
print('Processed data saved.')

In [ ]:
# Viz
fig = px.line(df, x='date', y='ridership', title='Ridership Trends')
fig.show()

fig2 = px.box(df, x='dayofweek', y='ridership')
fig2.show()